# OBD-II Data Cleaning and Preprocessing

Robust, loss-minimizing preprocessing for multi-trip OBD-II time-series data.


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Reproducibility for deterministic runs
np.random.seed(42)


In [2]:
RAW_FOLDER = Path("../data/raw/OBD-II-Dataset")
OUTPUT_FILE = Path("../data/processed/clean_obd_data.csv")
csv_files = sorted(RAW_FOLDER.glob("*.csv"))
print(f"Total files discovered: {len(csv_files)}")
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_FOLDER.resolve()}")


Total files discovered: 81


In [3]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    clean = []
    for col in df.columns:
        c = str(col).strip().lower()
        c = re.sub(r"[^a-z0-9]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        clean.append(c)
    df.columns = clean
    return df

RENAME_MAP = {
    "vehicle_speed_sensor_km_h": "speed",
    "engine_rpm_rpm": "rpm",
    "absolute_throttle_position": "throttle",
    "intake_manifold_absolute_pressure_kpa": "map",
    "air_flow_rate_from_mass_flow_sensor_g_s": "maf",
    "accelerator_pedal_position_d": "pedal_d",
    "accelerator_pedal_position_e": "pedal_e",
    "engine_coolant_temperature_c": "coolant_temp",
    "intake_air_temperature_c": "intake_temp",
    "ambient_air_temperature_c": "ambient_temp"
}

def convert_time(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if not text:
        return np.nan
    try:
        if text.replace(".", "", 1).isdigit():
            return float(text)
    except Exception:
        return np.nan
    parts = text.split(":")
    try:
        if len(parts) == 3:
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
        if len(parts) == 2:
            m, s = parts
            return int(m) * 60 + float(s)
    except ValueError:
        return np.nan
    return np.nan


In [4]:
# 1) Load and merge all trips
dfs = []
for i, file_path in enumerate(csv_files, start=1):
    df = pd.read_csv(file_path)
    df = normalize_columns(df)
    df = df.rename(columns=RENAME_MAP)
    df["trip_id"] = i
    dfs.append(df)
data = pd.concat(dfs, ignore_index=True)

# 2) Critical sensor validation
required_core = ["speed", "rpm", "throttle"]
missing_core = [c for c in required_core if c not in data.columns]
if missing_core:
    raise ValueError(f"Missing critical sensors: {missing_core}")

# 3) Time processing
time_candidates = [c for c in data.columns if c in {"time", "timestamp", "time_stamp"}]
if not time_candidates:
    raise KeyError("No time column found after normalization.")
time_col = time_candidates[0]

data["time_sec"] = data[time_col].apply(convert_time)
data = data.dropna(subset=["time_sec"]).copy()

# 4) Sorting and duplicate timestamp handling
data = data.sort_values(["trip_id", "time_sec"]).reset_index(drop=True)
data = data.drop_duplicates(subset=["trip_id", "time_sec"]).copy()

data["time"] = data.groupby("trip_id")["time_sec"].transform(lambda x: x - x.min())
data["dt"] = data.groupby("trip_id")["time_sec"].diff()
data = data[(data["dt"].isna()) | (data["dt"] > 0)].copy()

# Adaptive, data-driven gap detection
median_dt = data["dt"].median()
if pd.isna(median_dt) or median_dt <= 0:
    median_dt = 1.0
gap_threshold = median_dt * 3
data["gap_flag"] = (data["dt"] > gap_threshold).astype("int8")
print(f"Adaptive gap threshold: {gap_threshold:.6f}")


Adaptive gap threshold: 0.270000


In [5]:
# 5) Interpolation with strict sensor column control
expected_sensors = [
    "speed", "rpm", "throttle", "map",
    "maf", "pedal_d",
    "coolant_temp", "intake_temp", "ambient_temp"
]

sensor_cols = [c for c in expected_sensors if c in data.columns]
if not sensor_cols:
    raise ValueError("No expected sensors found in input data.")

data[sensor_cols] = data[sensor_cols].astype(float)

data[sensor_cols] = data.groupby("trip_id")[sensor_cols].transform(
    lambda x: x.interpolate(limit=3, limit_direction="both").ffill().bfill()
)

# Warning system for low-information (constant) signals
for col in sensor_cols:
    const_ratio = data.groupby("trip_id")[col].std().eq(0).mean()
    if const_ratio > 0.3:
        print(f"Warning: {col} has many constant segments ({const_ratio:.2%})")

# 6) Outlier clipping (unchanged fixed ranges)
clip_bounds = {
    "speed": (0, 200), "rpm": (0, 8000), "throttle": (0, 100), "map": (0, 300),
    "coolant_temp": (-40, 150), "intake_temp": (-40, 100), "ambient_temp": (-40, 60)
}
for col, (lo, hi) in clip_bounds.items():
    if col in data.columns:
        data[col] = data[col].clip(lo, hi)


In [6]:
# 7) Final schema order, validation, and exports
final_cols = ["trip_id", "time", "gap_flag"] + sensor_cols
data = data[final_cols].copy().sort_values(["trip_id", "time"]).reset_index(drop=True)

for col in data.select_dtypes(include=["float64"]):
    data[col] = pd.to_numeric(data[col], downcast="float")
for col in data.select_dtypes(include=["int64"]):
    data[col] = pd.to_numeric(data[col], downcast="integer")

rows_per_trip = data.groupby("trip_id").size().describe()
monotonic_ok = data.groupby("trip_id")["time"].apply(lambda s: s.is_monotonic_increasing).all()
dup_time_count = data.duplicated(subset=["trip_id", "time"]).sum()
negative_time_count = (data["time"] < 0).sum()
missing_total = int(data.isna().sum().sum())

print(rows_per_trip)
print(f"Time monotonic per trip: {monotonic_ok}")
print(f"Duplicated (trip_id, time): {dup_time_count}")
print(f"Negative time rows: {negative_time_count}")
print(f"Total missing values: {missing_total}")

assert monotonic_ok
assert dup_time_count == 0
assert negative_time_count == 0
assert missing_total == 0
assert data[sensor_cols].isna().sum().sum() == 0, "NaNs found in sensor columns"
assert not np.isinf(data[sensor_cols].to_numpy()).any(), "Infinite values found"

# 8) Trip summary export for analytics support
trip_summary = data.groupby("trip_id").agg({
    "time": ["min", "max"],
    "speed": ["mean", "std"],
    "rpm": ["mean", "std"]
})
trip_summary.to_csv("../data/processed/trip_summary.csv")
print("Trip summary saved.")

# 9) Save cleaned full dataset
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned full dataset: {OUTPUT_FILE}")
print(f"Final shape: {data.shape}")


count       81.000000
mean     33257.086420
std      16913.365097
min       6829.000000
25%      19645.000000
50%      32097.000000
75%      41611.000000
max      86654.000000
dtype: float64
Time monotonic per trip: True
Duplicated (trip_id, time): 0
Negative time rows: 0
Total missing values: 0
Trip summary saved.
Saved cleaned full dataset: ..\data\processed\clean_obd_data.csv
Final shape: (2693824, 12)


## Optional Debug Section (Not Part of Main Pipeline)

Sampling is intentionally removed from the production preprocessing pipeline.


In [7]:
ENABLE_DEBUG_SAMPLING = False
if ENABLE_DEBUG_SAMPLING:
    sample_fraction = 0.25
    samples = []
    for _, trip in data.groupby("trip_id"):
        trip = trip.sort_values("time")
        n = len(trip)
        sample_size = max(1, int(n * sample_fraction))
        if n <= sample_size:
            samples.append(trip)
            continue
        start = np.random.randint(0, n - sample_size + 1)
        samples.append(trip.iloc[start:start + sample_size])
    debug_sample = pd.concat(samples, ignore_index=True)
    print(f"Debug sample rows: {len(debug_sample):,}")
else:
    print("Debug sampling disabled.")


Debug sampling disabled.


## Final Summary

- Missing values are handled using per-trip interpolation (limit=3), then forward/backward fill.
- Irregular sampling is preserved using gap_flag instead of deleting rows.
- Realistic clipping reduces outlier/sensor noise impact.
- Sampling is removed from the production preprocessing pipeline to avoid bias.
- Final output is full, sorted, no-missing, and ML-ready.
